In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [22]:
# build the dataset
block_sizes = 3 # context length: how many characters do we take to predict the next one?
X, Y =[], []
for w in words[:5]:
    print(w)
    context = [0] * block_sizes
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append
        
X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [23]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [24]:
C = torch.randn((27, 2))

In [25]:
C

tensor([[ 0.0346,  0.8554],
        [ 0.2100,  1.9723],
        [-0.2954,  0.5485],
        [-0.7398, -1.1314],
        [ 0.3162,  1.2658],
        [-0.0454, -0.6572],
        [-1.8910, -0.2765],
        [-0.5670,  0.7136],
        [ 0.5030,  1.1027],
        [-0.0186, -0.6196],
        [ 0.2714,  1.0041],
        [ 1.2412,  0.5178],
        [-0.3009,  0.1313],
        [-0.0458, -1.4951],
        [ 0.0454,  0.8604],
        [-0.2724, -0.0811],
        [ 0.8075,  0.5351],
        [ 0.1961, -2.1520],
        [-1.6209,  1.1469],
        [-0.6971, -0.5120],
        [-1.4557, -1.1599],
        [-0.5108,  0.4783],
        [-0.9804,  2.7700],
        [-0.3902,  2.2165],
        [ 1.4759, -0.1286],
        [ 0.7621,  0.4682],
        [ 0.4895, -1.2692]])

In [26]:
C[5]

tensor([-0.0454, -0.6572])

In [27]:
F.one_hot(5, num_classes=27)

TypeError: one_hot(): argument 'input' (position 1) must be Tensor, not int

In [28]:
F.one_hot(torch.tensor(5), num_classes=27)

tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0])

In [29]:
F.one_hot(torch.tensor(5), num_classes=27).dtype

torch.int64

In [30]:
# same type as C, so we can multiply
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([-0.0454, -0.6572])

In [31]:
C[[5, 6, 7]]

tensor([[-0.0454, -0.6572],
        [-1.8910, -0.2765],
        [-0.5670,  0.7136]])

In [32]:
C[X]

tensor([[[ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [ 0.0346,  0.8554]],

        [[ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [-0.0454, -0.6572]],

        [[ 0.0346,  0.8554],
         [-0.0454, -0.6572],
         [-0.0458, -1.4951]],

        [[-0.0454, -0.6572],
         [-0.0458, -1.4951],
         [-0.0458, -1.4951]],

        [[-0.0458, -1.4951],
         [-0.0458, -1.4951],
         [ 0.2100,  1.9723]],

        [[ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [ 0.0346,  0.8554]],

        [[ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [-0.2724, -0.0811]],

        [[ 0.0346,  0.8554],
         [-0.2724, -0.0811],
         [-0.3009,  0.1313]],

        [[-0.2724, -0.0811],
         [-0.3009,  0.1313],
         [-0.0186, -0.6196]],

        [[-0.3009,  0.1313],
         [-0.0186, -0.6196],
         [-0.9804,  2.7700]],

        [[-0.0186, -0.6196],
         [-0.9804,  2.7700],
         [-0.0186, -0.6196]],

        [[-0.9804,  2

In [33]:
X[13, 2]

tensor(1)

In [34]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [35]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [36]:
emb @ W1 + b1

RuntimeError: mat1 and mat2 shapes cannot be multiplied (96x2 and 6x100)

In [39]:
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape

torch.Size([32, 6])

In [40]:
torch.unbind(emb, 1)

(tensor([[ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [-0.0454, -0.6572],
         [-0.0458, -1.4951],
         [ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [-0.2724, -0.0811],
         [-0.3009,  0.1313],
         [-0.0186, -0.6196],
         [-0.9804,  2.7700],
         [ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [ 0.2100,  1.9723],
         [ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [-0.0186, -0.6196],
         [-0.6971, -0.5120],
         [ 0.2100,  1.9723],
         [-0.2954,  0.5485],
         [-0.0454, -0.6572],
         [-0.3009,  0.1313],
         [ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [-0.6971, -0.5120],
         [-0.2724, -0.0811],
         [ 0.8075,  0.5351],
         [ 0.5030,  1.1027]]),
 tensor([[ 0.0346,  0.8554],
         [ 0.0346,  0.8554],
         [-0

In [41]:
len(torch.unbind(emb, 1))

3

In [43]:
torch.unbind(emb, 1)[0].shape

torch.Size([32, 2])

In [44]:
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([32, 6])

In [45]:
torch.cat(torch.unbind(emb, 1), 1)

tensor([[ 0.0346,  0.8554,  0.0346,  0.8554,  0.0346,  0.8554],
        [ 0.0346,  0.8554,  0.0346,  0.8554, -0.0454, -0.6572],
        [ 0.0346,  0.8554, -0.0454, -0.6572, -0.0458, -1.4951],
        [-0.0454, -0.6572, -0.0458, -1.4951, -0.0458, -1.4951],
        [-0.0458, -1.4951, -0.0458, -1.4951,  0.2100,  1.9723],
        [ 0.0346,  0.8554,  0.0346,  0.8554,  0.0346,  0.8554],
        [ 0.0346,  0.8554,  0.0346,  0.8554, -0.2724, -0.0811],
        [ 0.0346,  0.8554, -0.2724, -0.0811, -0.3009,  0.1313],
        [-0.2724, -0.0811, -0.3009,  0.1313, -0.0186, -0.6196],
        [-0.3009,  0.1313, -0.0186, -0.6196, -0.9804,  2.7700],
        [-0.0186, -0.6196, -0.9804,  2.7700, -0.0186, -0.6196],
        [-0.9804,  2.7700, -0.0186, -0.6196,  0.2100,  1.9723],
        [ 0.0346,  0.8554,  0.0346,  0.8554,  0.0346,  0.8554],
        [ 0.0346,  0.8554,  0.0346,  0.8554,  0.2100,  1.9723],
        [ 0.0346,  0.8554,  0.2100,  1.9723, -0.9804,  2.7700],
        [ 0.2100,  1.9723, -0.9804,  2.7

In [46]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [47]:
a.shape

torch.Size([18])

In [48]:
a.view(3, 3, 2)

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]],

        [[12, 13],
         [14, 15],
         [16, 17]]])

In [49]:
a.storage()

/var/folders/j7/2lnv016s5qn2r5lcrf2bk7080000gn/T/ipykernel_52920/2905276573.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [51]:
emb.shape

torch.Size([32, 3, 2])

In [52]:
emb.view(32, 6) == torch.cat(torch.unbind(emb, 1), 1)

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [53]:
h = emb.view(32, 6) @ W1 + b1

In [54]:
h

tensor([[ 1.4790, -0.9113,  2.1894,  ..., -4.6323, -1.2515,  0.4419],
        [ 2.1309, -0.5825,  2.0834,  ..., -4.1444,  0.8478,  0.8561],
        [ 3.4606,  2.8390, -0.2336,  ...,  0.3531,  1.9269,  1.0404],
        ...,
        [ 0.8247, -0.6473,  0.7708,  ..., -3.4688, -1.6325, -1.4946],
        [ 2.6569, -1.6300,  1.8249,  ..., -3.5211,  0.3335,  0.6669],
        [ 2.8931,  1.7035,  0.3472,  ..., -0.6436, -3.0980,  0.5385]])

In [59]:
h.shape

torch.Size([32, 100])

In [55]:
emb.view(-1, 6) @ W1 + b1

tensor([[ 1.4790, -0.9113,  2.1894,  ..., -4.6323, -1.2515,  0.4419],
        [ 2.1309, -0.5825,  2.0834,  ..., -4.1444,  0.8478,  0.8561],
        [ 3.4606,  2.8390, -0.2336,  ...,  0.3531,  1.9269,  1.0404],
        ...,
        [ 0.8247, -0.6473,  0.7708,  ..., -3.4688, -1.6325, -1.4946],
        [ 2.6569, -1.6300,  1.8249,  ..., -3.5211,  0.3335,  0.6669],
        [ 2.8931,  1.7035,  0.3472,  ..., -0.6436, -3.0980,  0.5385]])

In [56]:
emb.view(emb.shape[0], 6) @ W1 + b1

tensor([[ 1.4790, -0.9113,  2.1894,  ..., -4.6323, -1.2515,  0.4419],
        [ 2.1309, -0.5825,  2.0834,  ..., -4.1444,  0.8478,  0.8561],
        [ 3.4606,  2.8390, -0.2336,  ...,  0.3531,  1.9269,  1.0404],
        ...,
        [ 0.8247, -0.6473,  0.7708,  ..., -3.4688, -1.6325, -1.4946],
        [ 2.6569, -1.6300,  1.8249,  ..., -3.5211,  0.3335,  0.6669],
        [ 2.8931,  1.7035,  0.3472,  ..., -0.6436, -3.0980,  0.5385]])

In [57]:
h = torch.tanh(emb.view(emb.shape[0], 6) @ W1 + b1)

In [58]:
h.shape

torch.Size([32, 100])

In [60]:
b1.shape

torch.Size([100])

In [61]:
(emb.view(emb.shape[0], 6) @ W1).shape

torch.Size([32, 100])

In [62]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [63]:
logits = h @ W2 + b2

In [64]:
logits.shape

torch.Size([32, 27])

In [65]:
counts = logits.exp()

In [66]:
prob = counts / counts.sum(1, keepdim=True)

In [67]:
prob.shape

torch.Size([32, 27])

In [68]:
prob

tensor([[1.3583e-05, 2.8942e-13, 5.8633e-14, 2.9236e-15, 1.0850e-08, 1.5636e-16,
         3.8349e-06, 4.5168e-02, 4.5172e-09, 1.4350e-14, 6.6016e-11, 3.6482e-14,
         4.7296e-10, 3.6255e-06, 7.4189e-06, 1.6135e-02, 5.2358e-13, 4.9355e-11,
         9.1345e-06, 1.1173e-08, 3.0067e-03, 1.0166e-12, 9.3565e-01, 1.8588e-11,
         4.9424e-08, 3.6253e-06, 2.8416e-11],
        [1.0001e-06, 4.0275e-14, 6.7917e-15, 1.1208e-10, 5.0898e-08, 3.3313e-13,
         5.3542e-04, 6.6303e-01, 1.9249e-11, 2.5509e-12, 1.0378e-09, 1.3708e-10,
         1.1997e-12, 6.3528e-06, 3.2588e-05, 1.3968e-02, 1.3575e-09, 2.9394e-12,
         9.6349e-06, 5.4842e-05, 3.4074e-06, 9.2468e-12, 3.1860e-01, 5.0659e-04,
         3.0412e-04, 2.9402e-03, 8.0834e-07],
        [5.0202e-09, 9.7262e-11, 4.3017e-15, 3.1756e-06, 1.1914e-08, 4.7684e-12,
         1.2113e-01, 3.6942e-03, 4.0252e-12, 1.8006e-13, 3.7657e-06, 3.4706e-10,
         8.6039e-15, 3.1828e-11, 3.4752e-05, 1.4361e-04, 1.7745e-12, 7.3040e-09,
         5.7618e-

In [69]:
prob[0].sum()

tensor(1.0000)

In [70]:
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [71]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [72]:
prob[torch.arange(32), Y]

tensor([1.5636e-16, 6.3528e-06, 3.1828e-11, 6.8422e-11, 1.4917e-07, 1.6135e-02,
        3.1590e-11, 2.3797e-15, 1.8693e-07, 1.3329e-05, 6.6262e-12, 1.1108e-04,
        2.8942e-13, 6.7866e-02, 5.3964e-06, 4.2156e-02, 1.4350e-14, 5.9210e-05,
        2.9515e-07, 2.1688e-04, 7.4665e-10, 1.1350e-13, 3.3899e-10, 8.6650e-13,
        7.8596e-03, 1.1173e-08, 1.1229e-01, 3.5672e-14, 7.7264e-06, 1.5759e-11,
        3.1732e-11, 8.7600e-05])

In [73]:
loss = -prob[torch.arange(32), Y].log().mean()

In [74]:
loss

tensor(17.9376)

In [75]:
prob[torch.arange(32), Y]

tensor([1.5636e-16, 6.3528e-06, 3.1828e-11, 6.8422e-11, 1.4917e-07, 1.6135e-02,
        3.1590e-11, 2.3797e-15, 1.8693e-07, 1.3329e-05, 6.6262e-12, 1.1108e-04,
        2.8942e-13, 6.7866e-02, 5.3964e-06, 4.2156e-02, 1.4350e-14, 5.9210e-05,
        2.9515e-07, 2.1688e-04, 7.4665e-10, 1.1350e-13, 3.3899e-10, 8.6650e-13,
        7.8596e-03, 1.1173e-08, 1.1229e-01, 3.5672e-14, 7.7264e-06, 1.5759e-11,
        3.1732e-11, 8.7600e-05])

In [76]:
# --- now mode respectable code ---
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [77]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [78]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [79]:
emb = C[X]
h = torch.tanh(emb.view(emb.shape[0], -1) @ W1 + b1)
logits = h @ W2 + b2
counts = logits.exp()
prob = counts / counts.sum(1, keepdim=True)
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(17.7697)

In [80]:
F.cross_entropy(logits, Y)

tensor(17.7697)

In [81]:
emb = C[X]
h = torch.tanh(emb.view(emb.shape[0], -1) @ W1 + b1)
logits = h @ W2 + b2
# counts = logits.exp()
# prob = counts / counts.sum(1, keepdim=True)
# loss = -prob[torch.arange(32), Y].log().mean()
loss = F.cross_entropy(logits, Y)
loss

tensor(17.7697)

In [82]:
logits = torch.tensor([-2, -3, 0, 5])
counts = logits.exp()
probs = counts /counts.sum()
probs

tensor([9.0466e-04, 3.3281e-04, 6.6846e-03, 9.9208e-01])

In [83]:
logits = torch.tensor([100, -3, 0, 5])
counts = logits.exp()
probs = counts /counts.sum()
probs

tensor([nan, 0., 0., 0.])

In [84]:
counts

tensor([       inf, 4.9787e-02, 1.0000e+00, 1.4841e+02])